In [ ]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — feature addition

Loads the **transformed** frames from `transformed_data/`:

- `DF_common_1` — asset-level reserve rates/indexes, keyed on `(time_bucket, asset)`
- `DF_common_final` — protocol-level 6h time series, keyed on `time_bucket`

In [21]:
# Load transformed data from transformed_data/
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import adv_validation as adv

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 10

DF_common_1 = pd.read_csv(DATA_DIR / "DF_common_1.csv")          # asset-level (time_bucket, asset)
DF_common_final = pd.read_csv(DATA_DIR / "DF_common_final.csv")  # protocol-level 6h series

DF_common_2 = pd.read_csv(DATA_DIR / "DF_common_2.csv")          # asset-level (time_bucket, asset)
DF_common_final_1 = pd.read_csv(DATA_DIR / "DF_common_final_1.csv")  # protocol-level 2h series

In [22]:
original_cols_final = list(DF_common_final.columns)
original_cols_common_1 = list(DF_common_1.columns)

new_cols_final = [col for col in DF_common_final_1.columns if col not in original_cols_final]
new_cols_1 = [col for col in DF_common_2.columns if col not in original_cols_common_1]

print(f"DF_common_final — {len(new_cols_final)} new columns:")
# for i, col in enumerate(new_cols_final, 1):
#     print(f"  {i}. {col}")

print(f"\nDF_common_1 — {len(new_cols_1)} new columns:")
# for i, col in enumerate(new_cols_1, 1):
#     print(f"  {i}. {col}")


DF_common_final — 50 new columns:

DF_common_1 — 24 new columns:


In [23]:
supply_withdrawal_columns = [
    "supply_tx_count", "withdrawal_tx_count", "unique_suppliers", "unique_withdraw_users",
    "supply_amount_value_usd", "supply_amount_value_eth",
    "withdrawal_amount_value_usd", "withdrawal_amount_value_eth",
    "net_liquidity_flow_usd", "net_liquidity_flow_eth",
    "supply_withdrawal_ratio", "liquidity_growth_rate",
    "avg_supply_size_usd", "avg_withdrawal_size_usd",
]

borrow_repay_columns = [
    "borrow_tx_count", "repay_tx_count", "variable_borrow_tx_count",
    "unique_borrowers", "unique_repayers",
    "borrow_amount_value_usd", "borrow_amount_value_eth",
    "repay_amount_value_usd", "repay_amount_value_eth",
    "net_borrow_demand_usd", "net_borrow_demand_eth",
    "borrow_repay_ratio",
    "avg_borrow_size_usd", "avg_borrow_size_eth",
    "avg_repay_size_usd", "avg_repay_size_eth",
    "borrow_growth", "debt_expansion_ratio",
]

collateral_columns = [
    "collateral_enabled_count", "collateral_disabled_count",
    "unique_collateral_enable_users", "unique_collateral_disable_users",
    "as_collateral_tx_count", "as_debt_tx_count",
    "collateral_usage_rate", "collateral_adoption_rate",
]

liquidation_columns = [
    "liquidation_tx_count", "unique_liquidated_users", "unique_liquidators",
    "liquidated_collateral_value_usd", "liquidated_collateral_value_eth",
    "liquidation_debt_covered_value_usd", "liquidation_debt_covered_value_eth",
    "liquidation_rate", "liquidation_volume_ratio_eth",
    "liquidation_severity_usd", "liquidation_severity_eth",
    "avg_liquidation_debt_usd", "avg_liquidation_debt_eth",
    "liquidator_concentration", "liquidation_user_ratio",
]

risk_columns = [
    "avg_total_collateral_base", "avg_total_debt_base", "avg_available_borrows_base",
    "avg_current_liquidation_threshold", "avg_ltv",
    "sampled_user_count", "account_data_call_count",
    "collateralization_ratio", "borrow_capacity_utilization",
    "remaining_borrow_capacity", "risk_buffer",
    "ltv_utilization", "distance_to_liquidation",
]

flashloan_columns = [
    "flashloan_tx_count", "unique_flashloan_initiators",
    "flashloan_amount_value_usd", "flashloan_amount_value_eth",
    "flashloan_premium_value_usd", "flashloan_premium_value_eth",
    "no_open_debt_flashloan_tx_count", "variable_flashloan_tx_count",
    "avg_flashloan_size_usd", "avg_flashloan_size_eth",
    "flashloan_fee_rate", "flashloan_usage_intensity",
    "flashloan_user_activity", "variable_debt_flashloan_ratio",
    "no_debt_flashloan_ratio",
]

user_behaviour_columns = [
    "borrower_participation_rate", "repayment_discipline",
    "supplier_activity", "borrower_activity",
]

market_wide_columns = [
    "total_activity", "user_activity",
    "protocol_turnover_usd", "protocol_turnover_eth",
    "leverage_indicator",
    "market_stress_index_usd", "market_stress_index_eth",
    "capital_efficiency",
]

In [24]:
groups = {
    "Supply & Withdrawal": supply_withdrawal_columns,
    "Borrow & Repay":      borrow_repay_columns,
    "Collateral":          collateral_columns,
    "Liquidation":         liquidation_columns,
    "Risk":                risk_columns,
    "Flashloan":           flashloan_columns,
    "User Behaviour":      user_behaviour_columns,
    "Market Wide":         market_wide_columns,
}

# for name, cols in groups.items():
#     print(f"\n{'='*40}\n{name}\n{'='*40}")
#     display(DF_common_final_1[["time_bucket"] + cols])

In [25]:
# Split DF_common_final_1 into one named DataFrame per metric group (time_bucket kept as the key)
group_frames = {}
for _name, _cols in groups.items():
    _missing = [c for c in _cols if c not in DF_common_final_1.columns]
    if _missing:
        print(f"⚠ {_name}: missing {_missing}")
    _present = [c for c in _cols if c in DF_common_final_1.columns]
    group_frames[_name] = DF_common_final_1[["time_bucket"] + _present].copy()

# expose each group as its own named DataFrame
supply_withdrawal_df = group_frames["Supply & Withdrawal"]
borrow_repay_df      = group_frames["Borrow & Repay"]
collateral_df        = group_frames["Collateral"]
liquidation_df       = group_frames["Liquidation"]
risk_df              = group_frames["Risk"]
flashloan_df         = group_frames["Flashloan"]
user_behaviour_df    = group_frames["User Behaviour"]
market_wide_df       = group_frames["Market Wide"]

for _name, _frame in group_frames.items():
    print(f"{_name:22s} {_frame.shape[0]} rows x {_frame.shape[1]} cols")

Supply & Withdrawal    4380 rows x 15 cols
Borrow & Repay         4380 rows x 19 cols
Collateral             4380 rows x 9 cols
Liquidation            4380 rows x 16 cols
Risk                   4380 rows x 14 cols
Flashloan              4380 rows x 16 cols
User Behaviour         4380 rows x 5 cols
Market Wide            4380 rows x 9 cols


In [38]:
stats = adv.statistical_validation(DF_common_final_1, table_name="DF_common_final_1", save=False)

with pd.option_context("display.max_rows", None):   # show every column-row, no '..' collapse
    stats["p95_p05_ratio"] = stats["p95"] / stats["p05"].replace(0, np.nan)

    stats["nonparam_skew"] = (stats["mean"] - stats["median"]) / stats["std"].replace(0, np.nan)

    # if {"std", "n_checked"} <= have:
    stats["sem"] = stats["std"] / np.sqrt(stats["n_checked"])

    # 3) robust CV = mad/median  and  iqr/median            [needs: mad|iqr, median]
    stats["robust_cv_mad"] = stats["mad"] / stats["median"].replace(0, np.nan)

    stats["robust_cv_iqr"] = stats["iqr"] / stats["median"].replace(0, np.nan)

# 4) quartile coeff. of dispersion = (q75-q25)/(q75+q25)  [needs: q25, q75]

    stats["qcd"] = (stats["q75"] - stats["q25"]) / (stats["q75"] + stats["q25"]).replace(0, np.nan)

# 5) tail ratio = p95/p05  (and p99/p01)                [needs: p95/p05, p99/p01]

    stats["tail_ratio_95_05"] = stats["p95"] / stats["p05"].replace(0, np.nan)

    stats["tail_ratio_99_01"] = stats["p99"] / stats["p01"].replace(0, np.nan)

    stats["tail_ratio_99_95"] = stats["p99"] / stats["p95"].replace(0, np.nan)

    stats["robust_cv_99th"] = (stats["p99"] - stats["p95"])/ stats["p99"].replace(0, np.nan)

    stats["robust_cv_99th_95_spread"] = (stats["p99"] - stats["p95"])/ (stats["p99"] + stats["p95"]).replace(0, np.nan)

    display(stats[["column",
        #          "null_pct", "n_zero", "zero_pct", "negative_pct",
        # "n_sentinel", "mean",
        "cv",
        # "std",
        # "min", 
        # "p01", "p05", 
        # "q25", "median", "q75",
        "p95", "p99", #"max",
        #  "iqr",
        #  "mad",
        "skewness", 
         "excess_kurtosis",
         "outlier_iqr_pct", "outlier_mad_pct",
        # "heavy_tailed", "success", "n_outliers_iqr",  "n_outliers_mad",
        # "nonparam_skew", "sem", 
        # "robust_cv_mad", "robust_cv_iqr",
        #    "qcd",
             "tail_ratio_95_05", "tail_ratio_99_01", "tail_ratio_99_95",
             "robust_cv_99th",
             "robust_cv_99th_95_spread"
        ]])

,column,cv,p95,p99,skewness,excess_kurtosis,outlier_iqr_pct,outlier_mad_pct,tail_ratio_95_05,tail_ratio_99_01,tail_ratio_99_95,robust_cv_99th,robust_cv_99th_95_spread
0,supply_tx_count,0.532505,3.470000e+02,5.552100e+02,3.392236,22.072088,4.8858,3.6758,3.988506e+00,8.286716e+00,1.600029,0.375011,0.230778
1,withdrawal_tx_count,0.575709,3.000000e+02,4.862100e+02,3.597868,25.331848,4.9087,3.7443,4.411765e+00,9.533529e+00,1.620700,0.382983,0.236845
2,unique_suppliers,0.475759,1.720000e+02,2.664200e+02,3.633662,28.164152,4.0411,2.5571,3.440000e+00,6.831282e+00,1.548953,0.354403,0.215364
3,unique_withdraw_users,0.541349,1.540500e+02,2.532100e+02,4.865069,49.673533,4.3379,3.0594,3.757317e+00,7.912813e+00,1.643687,0.391612,0.243481
4,supply_amount_value_usd,1.914503,6.343127e+08,1.249189e+09,12.201314,220.560140,7.0776,9.9543,4.263439e+01,1.947249e+02,1.969359,0.492221,0.326454
5,supply_amount_value_eth,1.537185,2.541312e+05,4.164420e+05,6.760463,84.307378,9.0868,12.0776,5.166079e+01,1.998047e+02,1.638689,0.389756,0.242048
6,withdrawal_amount_value_usd,1.959022,6.323593e+08,1.293650e+09,12.141956,217.563012,6.7808,10.4338,4.493345e+01,1.968091e+02,2.045752,0.511182,0.343348
7,withdrawal_amount_value_eth,1.576295,2.526450e+05,4.536389e+05,6.765406,82.845797,8.9269,12.5114,5.332480e+01,2.048702e+02,1.795558,0.443070,0.284579
8,borrow_tx_count,0.492772,2.010000e+02,2.743300e+02,1.897450,8.215089,3.1136,1.9689,4.466667e+00,8.397000e+00,1.364826,0.267306,0.154272
9,repay_tx_count,0.808015,1.970000e+02,3.706600e+02,5.314877,49.264260,6.2958,5.3342,5.969697e+00,1.482640e+01,1.881523,0.468516,0.305923


## Column priority map (shared across notebooks)

A `{column: score}` map you tune as the EDA progresses. It is **file-backed**
(`column_priority.json`) so it is **global to this directory** and **shared across
notebooks**: an edit here is visible in any other notebook the next time that notebook
calls `eda.get_priorities()` / `eda.get_priority()` — each call re-reads the file
(separate kernels can't share memory, so *re-read = refresh*). Logic lives in `EDA.py`.

In [ ]:
# --- Shared column-priority map (persisted in column_priority.json; functions in EDA.py) ---
import EDA as eda

# Seed every feature column at priority 0 (existing scores are KEPT), then view the map.
all_feature_cols = [c for cols in groups.values() for c in cols]
eda.register_columns(all_feature_cols, default=0)

priority_df = eda.get_priorities()   # column -> score, highest first; re-reads the shared file
display(priority_df)

In [ ]:
# Adjust priorities as you go — each call writes the shared file (other notebooks see it on their next read):
# eda.increase_priority("borrow_amount_value_usd", by=3)   # raise
# eda.decrease_priority("flashloan_tx_count", by=1)        # lower
# eda.set_priority("avg_ltv", 5)                           # absolute score
# eda.update_priorities({"liquidation_rate": 4, "net_borrow_demand_usd": 2})  # bulk set
# eda.remove_priority("some_col")                          # drop a column
# eda.register_columns(DF_common_1.columns, default=0)     # also seed the asset-level cols
eda.get_priorities()       # always the latest map (re-reads column_priority.json)